In [49]:
import pandas as pd

# 1. Load both sheets
file_path = "knn_imbalanced_dataset_Q2.xlsx"
train_df = pd.read_excel(file_path, sheet_name="Train")
test_df  = pd.read_excel(file_path, sheet_name="Test")

train_df.head()

,Sample_ID,Temperature_C,Pressure_bar,Vibration_mm_s,Current_A,Acoustic_dB,Class
0,S0630,65.86,101.19,1.775,60.77,16.53,Fault
1,S0500,69.05,108.78,0.605,38.71,15.08,Normal
2,S0136,51.40,148.59,0.863,36.94,9.67,Normal
3,S0481,55.77,119.80,1.051,37.81,9.84,Normal
4,S0091,49.48,124.19,1.093,45.38,10.14,Normal


In [50]:
features = ["Temperature_C", "Pressure_bar", "Vibration_mm_s", "Current_A", "Acoustic_dB"]
target = "Class"


In [51]:
X_train = train_df[features]
y_train = train_df[target]

In [52]:
X_test = test_df[features]
y_test = test_df[target]

In [53]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [54]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

best_k, best_score = None, 0
for k in [3,5,7,9,11,15,21]:
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=cv, scoring='f1_macro')
    mean_score = np.mean(scores)
    if mean_score > best_score:
        best_k, best_score = k, mean_score

print("Best k:", best_k, "with F1:", best_score)

Best k: 3 with F1: 0.9519000933294972


In [55]:
knn = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean')
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)

y_pred.shape

(210,)

In [56]:
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, balanced_accuracy_score

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Precision Score:\n", precision_score(y_test, y_pred, pos_label="Normal"))
print("Recall Score:\n", recall_score(y_test, y_pred, pos_label="Normal"))
print("F1 Score:\n", f1_score(y_test, y_pred, pos_label="Normal"))
print("Balanced Accuracy Score:\n", balanced_accuracy_score(y_test, y_pred))

print("\nClassification Report:\n", classification_report(y_test, y_pred))

Confusion Matrix:
 [[ 24   6]
 [  0 180]]
Precision Score:
 0.967741935483871
Recall Score:
 1.0
F1 Score:
 0.9836065573770492
Balanced Accuracy Score:
 0.9

Classification Report:
               precision    recall  f1-score   support

       Fault       1.00      0.80      0.89        30
      Normal       0.97      1.00      0.98       180

    accuracy                           0.97       210
   macro avg       0.98      0.90      0.94       210
weighted avg       0.97      0.97      0.97       210



In [ ]:
from imblearn.over_sampling import RandomOverSampler

# Oversample minority class
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_train_scaled, y_train)

# Train KNN
knn_ros = KNeighborsClassifier(n_neighbors=7, metric="euclidean")
knn_ros.fit(X_resampled, y_resampled)

# Predict
y_pred_ros = knn_ros.predict(X_test_scaled)
print("Oversampling Confusion Matrix:\n", confusion_matrix(y_test, y_pred_ros))
print("F1 Score:\n", f1_score(y_test, y_pred_ros, pos_label="Normal"))
print("Balanced Accuracy Score:\n", balanced_accuracy_score(y_test, y_pred_ros))

print("\nOversampling Classification Report:\n", classification_report(y_test, y_pred_ros))

Oversampling Confusion Matrix:
 [[ 29   1]
 [  6 174]]
F1 Score:
 0.9802816901408451
Balanced Accuracy Score:
 0.9666666666666667

Oversampling Classification Report:
               precision    recall  f1-score   support

       Fault       0.83      0.97      0.89        30
      Normal       0.99      0.97      0.98       180

    accuracy                           0.97       210
   macro avg       0.91      0.97      0.94       210
weighted avg       0.97      0.97      0.97       210



: 

In [59]:
# Distance-weighted KNN
knn_weighted = KNeighborsClassifier(n_neighbors=7, metric="euclidean", weights="distance")
knn_weighted.fit(X_train_scaled, y_train)

# Predict
y_pred_weighted = knn_weighted.predict(X_test_scaled)
print("Distance-weighted Confusion Matrix:\n", confusion_matrix(y_test, y_pred_weighted))
print("F1 Score:\n", f1_score(y_test, y_pred_weighted, pos_label="Normal"))
print("Balanced Accuracy Score:\n", balanced_accuracy_score(y_test, y_pred_weighted))

print("\nDistance-weighted Classification Report:\n", classification_report(y_test, y_pred_weighted))

Distance-weighted Confusion Matrix:
 [[ 25   5]
 [  0 180]]
F1 Score:
 0.9863013698630136
Balanced Accuracy Score:
 0.9166666666666667

Distance-weighted Classification Report:
               precision    recall  f1-score   support

       Fault       1.00      0.83      0.91        30
      Normal       0.97      1.00      0.99       180

    accuracy                           0.98       210
   macro avg       0.99      0.92      0.95       210
weighted avg       0.98      0.98      0.98       210

